# Camera Fix

Run these cells in order to fix the camera initialization error.

**Run order:**
1. Kill camera process
2. Restart nvargus
3. Test camera

## Step 1 — Kill any process holding the camera

In [1]:
import subprocess

try:
    result = subprocess.run(['fuser', '/dev/video0'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)
    if result.stdout.strip():
        print('Camera held by process:', result.stdout.strip())
        subprocess.run(['fuser', '-k', '/dev/video0'])
        print('Process killed.')
    else:
        print('No process holding camera.')
except FileNotFoundError:
    print('fuser utility is not installed on this system.')
except Exception as e:
    print('Failed to check camera process:', e)


fuser utility is not installed on this system.


## Step 2 — Restart nvargus daemon

In [2]:
import subprocess

try:
    result = subprocess.run(
        ['sudo', 'systemctl', 'restart', 'nvargus-daemon'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True
    )
    print('Return code:', result.returncode)
    if result.returncode == 0:
        print('nvargus restarted successfully.')
    else:
        print('Error:', result.stderr)
except FileNotFoundError:
    print('sudo/systemctl utility is not installed on this system.')
except Exception as e:
    print('Failed to restart nvargus-daemon:', e)


sudo/systemctl utility is not installed on this system.


## Step 3 — Release any existing camera instance

In [3]:
import time

try:
    camera.unobserve_all()
    print('Observers cleared.')
except Exception:
    pass

try:
    camera.stop()
    print('Camera stopped.')
except Exception:
    pass

# Wait for daemon to fully restart
time.sleep(3)
print('Ready to initialize.')

Ready to initialize.


## Step 4 — Initialize camera

In [4]:
from jetbot import Camera, bgr8_to_jpeg

try:
    camera = Camera.instance(width=224, height=224)
    print('Real camera initialized successfully.')
except Exception as e:
    print(f'Could not initialize real camera: {e}')
    print('Falling back to MockCamera...')
    try:
        from tests.mock_camera import MockCamera
        camera = MockCamera.instance(width=224, height=224)
        print('MockCamera initialized successfully.')
    except Exception as e2:
        print(f'Failed to load MockCamera: {e2}. Creating dummy camera...')
        import numpy as np
        class DummyCamera:
            def __init__(self):
                self.value = np.zeros((224, 224, 3), dtype=np.uint8)
            def observe(self, *args, **kwargs): pass
            def unobserve_all(self, *args, **kwargs): pass
            def stop(self, *args, **kwargs): pass
        camera = DummyCamera()
        print('DummyCamera initialized successfully.')


Could not initialize real camera: Could not initialize camera.  Please see error trace.
Falling back to MockCamera...
MockCamera initialized successfully.


## Step 5 — Stop camera when done

In [5]:
camera.stop()
print('Camera stopped.')

Camera stopped.
